In [0]:
from pyspark.sql.functions import col, expr

In [0]:
checkpoint = '/Volumes/retail_store_dev/landing/retail_store/order_silver_checkpoint/'
checkpoint_fct = '/Volumes/retail_store_dev/landing/retail_store/fact_order_silver_checkpoint/'

In [0]:
%sql
USE CATALOG IDENTIFIER(:catalog);

In [0]:
bronze_df = spark.readStream \
    .table("bronze.orders")

bronze_df = bronze_df.withColumns({
    "order_date": expr("order_date::timestamp"),
    "employee_id": expr("employee_id::int"),
    "order_amount": expr("order_amount::double"),
    "pincode": expr("right(customer_address, 6)?::int"),
    "line_items": expr("from_json(line_items, 'array<struct<product_id: int, quantity: int>>')")
})

In [0]:
sQry = bronze_df.writeStream \
    .trigger(availableNow=True) \
        .format("delta") \
        .outputMode("append") \
            .option("checkpointLocation", checkpoint) \
    .table("silver.orders_cleansed")

In [0]:
clean_order_df = spark.readStream \
    .format("delta") \
    .table("silver.orders_cleansed") 
clean_order_df = clean_order_df\
        .select(
            "order_id",
            "order_date",
            "employee_id",
            expr("pincode as customer_pincode"),
            "order_amount",
            expr("inline(line_items)"),            
            "load_date"
        )
sQryf = clean_order_df \
    .writeStream \
        .outputMode("append") \
            .format("delta") \
                .trigger(availableNow=True) \
                    .option("checkpointLocation", checkpoint_fct) \
                    .toTable("silver.fact_orders")